<a href="https://colab.research.google.com/github/likith2109/demo-repo/blob/main/p1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pennylane qiskit qiskit-machine-learning torch torchvision scikit-learn pandas numpy

  Using cached pennylane-0.45.1-py3-none-any.whl.metadata (10 kB)
  Using cached qiskit-2.5.1-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (14 kB)
  Using cached qiskit_machine_learning-0.9.0-py3-none-any.whl.metadata (13 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 8.7 MB

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/Heart Disease Dataset.csv')

# Display the first 5 rows
display(df.head())

,Age,Cholesterol,Maximum Heart Rate,Resting Blood Pressure,HeartDisease
0,54,206,108,110,0
1,51,227,154,94,1
2,63,187,144,140,0
3,42,209,173,120,1
4,44,226,169,120,1


In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.svm import SVC

# PyTorch & PennyLane
import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml

# Qiskit for QSVM
from qiskit.circuit.library import PauliFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [ ]:
def load_and_preprocess_data():
    # Load dataset
    df = pd.read_csv("Heart Disease Dataset.csv")

    # Select the 4 clinical features as specified
    features = ['Age', 'Cholesterol', 'Maximum Heart Rate', 'Resting Blood Pressure']
    X = df[features].values

    # Target column verified as 'HeartDisease'
    y = df['HeartDisease'].values

    # Min-Max Scaling
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # Angle Encoding mapping scales features to [0, pi]
    X_scaled = X_scaled * np.pi

    # Split the data (80% training, 20% testing)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = load_and_preprocess_data()
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

Training samples: 262, Testing samples: 66


In [ ]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    # Explicitly handle batching by iterating if necessary,
    # though PennyLane usually handles RX broadcasting.
    for i in range(n_qubits):
        qml.RX(inputs[..., i], wires=i)

    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

In [ ]:
class VQC(nn.Module):
    def __init__(self, n_layers=2):
        super(VQC, self).__init__()
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

    def forward(self, x):
        # Ensure the output has the shape (batch_size, 1)
        x = self.qlayer(x).reshape(-1, 1)
        return torch.sigmoid(x)

class HQNN(nn.Module):
    def __init__(self, n_layers=2):
        super(HQNN, self).__init__()
        self.fc1 = nn.Linear(4, 4)
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)
        self.fc2 = nn.Linear(1, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        # Reshape to (batch_size, 1) before passing to the final linear layer
        x = self.qlayer(x).reshape(-1, 1)
        x = self.fc2(x)
        return torch.sigmoid(x)

In [ ]:
def train_and_evaluate_torch_model(model, X_train, y_train, X_test, y_test, epochs=20):
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    criterion = nn.BCELoss()
    # Using Adam optimizer as required
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    start_time = time.time()

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

    train_time = time.time() - start_time

    # Testing
    test_start_time = time.time()
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).numpy().flatten()
        preds = (preds > 0.5).astype(int)
    test_time = time.time() - test_start_time

    metrics = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1-Score": f1_score(y_test, preds, zero_division=0),
        "Training Time (s)": train_time,
        "Testing Time (s)": test_time
    }
    return metrics

def run_qsvm(X_train, y_train, X_test, y_test):
    # Using PauliFeatureMap for Qiskit-based Quantum Kernel Evaluation
    feature_map = PauliFeatureMap(feature_dimension=4, reps=2, paulis=['Z'])
    qkernel = FidelityQuantumKernel(feature_map=feature_map)

    # Initialize traditional SVC using the quantum kernel
    qsvm = SVC(kernel=qkernel.evaluate)

    start_time = time.time()
    qsvm.fit(X_train, y_train)
    train_time = time.time() - start_time

    test_start_time = time.time()
    preds = qsvm.predict(X_test)
    test_time = time.time() - test_start_time

    metrics = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1-Score": f1_score(y_test, preds, zero_division=0),
        "Training Time (s)": train_time,
        "Testing Time (s)": test_time
    }
    return metrics

In [ ]:
results = {}

# 1. Variational Quantum Classifier (VQC)
print("Training VQC...")
vqc_model = VQC(n_layers=2)
results["VQC"] = train_and_evaluate_torch_model(vqc_model, X_train, y_train, X_test, y_test)

# 2. Quantum Neural Network (QNN)
print("Training QNN...")
qnn_model = VQC(n_layers=3)
results["QNN"] = train_and_evaluate_torch_model(qnn_model, X_train, y_train, X_test, y_test)

# 3. Hybrid Quantum Neural Network (HQNN)
print("Training HQNN...")
hqnn_model = HQNN(n_layers=2)
results["HQNN"] = train_and_evaluate_torch_model(hqnn_model, X_train, y_train, X_test, y_test)

# 4. Quantum Support Vector Machine (QSVM)
print("Training QSVM (Qiskit)... (This may take a few minutes)")
results["QSVM"] = run_qsvm(X_train, y_train, X_test, y_test)

# Print Results Comparison Table
print("\n" + "="*80)
print("PERFORMANCE EVALUATION METRICS COMPARISON")
print("="*80)
df_results = pd.DataFrame(results).T
print(df_results.to_string())
print("="*80)

Training VQC...
Training QNN...
Training HQNN...
Training QSVM (Qiskit)... (This may take a few minutes)


/tmp/ipykernel_944/2841967637.py:42: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation.pauli_feature_map.PauliFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the pauli_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = PauliFeatureMap(feature_dimension=4, reps=2, paulis=['Z'])



PERFORMANCE EVALUATION METRICS COMPARISON
      Accuracy  Precision    Recall  F1-Score  Training Time (s)  Testing Time (s)
VQC   0.757576   0.740741  0.689655  0.714286           0.515317          0.009989
QNN   0.772727   0.818182  0.620690  0.705882           0.564330          0.013714
HQNN  0.530303   0.480769  0.862069  0.617284           0.494859          0.009858
QSVM  0.636364   0.586207  0.586207  0.586207         194.223585         96.125657
